# 03 — Modelagem Baseline

**Objetivo:** Decidir empiricamente a estratégia de balanceamento (4 testadas) e comparar 7 modelos por PR-AUC com StratifiedKFold, elegendo os finalistas (XGBoost e RandomForest) para o tuning.

---

**Roteiro:**

1. Setup
2. Carregamento de dados
3. Comparação de estratégias de balanceamento
4. Comparação de modelos (com class_weight)
5. Visualização — gráfico de barras por modelos 
6. Persistir artefato
7. Fechamento


### Etapa 1 - Setup

In [1]:
%load_ext autoreload
%autoreload 2

import time

import joblib
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

from src.config import CONFIG, CAMINHOS
from src.viz_config import aplicar_tema_seaborn, PALETTE, CORES

from imblearn.pipeline import Pipeline as ImbPipeline
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from lightgbm import LGBMClassifier
from sklearn.calibration import CalibratedClassifierCV
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import make_scorer, average_precision_score
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.neighbors import KNeighborsClassifier
from sklearn.pipeline import Pipeline as SkPipeline
from sklearn.preprocessing import RobustScaler
from sklearn.svm import LinearSVC
from sklearn.tree import DecisionTreeClassifier
from xgboost import XGBClassifier

DADOS_PRO    = CAMINHOS.dados_processed
MODELS_DIR   = CAMINHOS.modelos
FIGURAS_DIR  = CAMINHOS.figures
RANDOM_STATE = CONFIG["dados"]["random_state"]

aplicar_tema_seaborn()

print("\n✅ Setup configurado!\n")

print(f"SEED             : {RANDOM_STATE}")
print(f"FIGURAS          : {FIGURAS_DIR}")
print(f"MODELOS          : {MODELS_DIR}")
print(f"DADOS PROCESSADOS: {DADOS_PRO}")


✅ Setup configurado!

SEED             : 42
FIGURAS          : C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\reports\figures
MODELOS          : C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\models
DADOS PROCESSADOS: C:\Users\jas_t\Repo\Portfolios\credit_card_fraud\data\processed


### Etapa 2 - Carregamento de dados

Esta etapa será organizada em quatro partes principais, garantindo um fluxo consistente e reprodutível para a avaliação dos modelos.

1. **Carregamento dos dados:**
   Leitura dos conjuntos de treino e teste salvos previamente em formato Parquet.

2. **Reconstrução do pré-processador:**
   Recriação da estrutura de pré-processamento definida no NB02, garantindo consistência entre as etapas do projeto.

3. **Definição dos scorers:**
   Configuração das métricas de avaliação, com **PR-AUC** como métrica primária devido ao forte desbalanceamento da variável alvo.

4. **Validação cruzada estratificada:**
   Aplicação de **StratifiedKFold** para preservar a proporção de fraudes em cada fold, tornando a avaliação mais estável e adequada ao problema.

In [2]:
X_train = pd.read_parquet(DADOS_PRO / "X_train.parquet")
X_test  = pd.read_parquet(DADOS_PRO / "X_test.parquet")

y_train = pd.read_parquet(DADOS_PRO / "y_train.parquet").iloc[:, 0]
y_test  = pd.read_parquet(DADOS_PRO / "y_test.parquet").iloc[:, 0]

print(f"Treino: {X_train.shape} · {y_train.sum()} fraudes")

Treino: (226980, 30) · 378 fraudes


In [3]:
V_COLS = [c for c in X_train.columns if c.startswith("V")]

preprocessor = ColumnTransformer(
    transformers=[
        ("amount", RobustScaler(), ["Amount"]),
        ("vs",     "passthrough",  V_COLS),
    ],
    remainder="drop",
    verbose_feature_names_out=False,
)

In [4]:
SCORERS = {
    "pr_auc":    "average_precision",
    "roc_auc":   "roc_auc",
    "recall":    "recall",
    "precision": "precision",
    "f1":        "f1",
}

print(f"Métrica primária: PR-AUC (average_precision)")

Métrica primária: PR-AUC (average_precision)


In [5]:
CV = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

print(f"CV: StratifiedKFold(5)")

CV: StratifiedKFold(5)


### Etapa 3 - Comparação de estratégias de balanceamento

Nesta etapa, o objetivo é isolar o impacto das estratégias de balanceamento na performance do modelo.

Para isso, o modelo será mantido fixo como **Logistic Regression**, permitindo que a comparação seja focada apenas no efeito do tratamento do desbalanceamento da classe alvo.

Serão avaliadas quatro estratégias:

1. **Sem balanceamento — controle**
   O modelo será treinado com a distribuição original das classes, servindo como baseline para comparação.

2. **`class_weight='balanced'`**
   A classe minoritária recebe maior peso durante o treinamento, sem alterar a quantidade original de registros.

3. **SMOTE puro**
   Geração de exemplos sintéticos da classe fraudulenta até atingir uma distribuição balanceada entre as classes.

4. **SMOTE moderado + undersampling**
   Estratégia híbrida em que o SMOTE eleva a proporção de fraudes para aproximadamente **10%**, seguido de undersampling da classe majoritária para ajustar o balanceamento final.

Essa comparação permite avaliar se o ganho de desempenho vem do balanceamento artificial dos dados ou apenas do ajuste de pesos no treinamento, sempre utilizando a mesma arquitetura de modelo para manter a análise controlada.

In [6]:
base_lr = LogisticRegression(max_iter=2_000, random_state=RANDOM_STATE)

In [7]:
estrategias = {
    "1. Sem balanceamento": SkPipeline([
        ("prep", preprocessor),
        ("clf",  base_lr),
    ]),
    "2. class_weight='balanced'": SkPipeline([
        ("prep", preprocessor),
        ("clf",  LogisticRegression(max_iter=2000, class_weight="balanced",
                                    random_state=RANDOM_STATE)),
    ]),
    "3. SMOTE": ImbPipeline([
        ("prep",  preprocessor),
        ("smote", SMOTE(random_state=RANDOM_STATE)),
        ("clf",   base_lr),
    ]),
    "4. SMOTE + Undersampling": ImbPipeline([
        ("prep",  preprocessor),
        ("smote", SMOTE(sampling_strategy=0.1, random_state=RANDOM_STATE)),
        ("under", RandomUnderSampler(sampling_strategy=0.5, random_state=RANDOM_STATE)),
        ("clf",   base_lr),
    ]),
}

In [8]:
print("═" * 72)
print("COMPARAÇÃO DE ESTRATÉGIAS DE BALANCEAMENTO (LogisticRegression)")
print("═" * 72)
print(f"   {'estratégia':<28}{'PR-AUC':>9}{'ROC-AUC':>9}{'recall':>9}{'precis.':>9}")
print("   " + "─" * 64)

linhas = []
for nome, pipe in estrategias.items():
    res = cross_validate(pipe, X_train, y_train, cv=CV, scoring=SCORERS, n_jobs=-1)
    linhas.append({
        "estrategia": nome,
        "pr_auc":    res["test_pr_auc"].mean(),
        "roc_auc":   res["test_roc_auc"].mean(),
        "recall":    res["test_recall"].mean(),
        "precision": res["test_precision"].mean(),
    })
    print(f"   {nome:<28}{linhas[-1]['pr_auc']:>9.3f}{linhas[-1]['roc_auc']:>9.3f}"
          f"{linhas[-1]['recall']:>9.3f}{linhas[-1]['precision']:>9.3f}")

res_bal = pd.DataFrame(linhas)
melhor = res_bal.loc[res_bal["pr_auc"].idxmax(), "estrategia"]
print(f"\n   🏆 Melhor PR-AUC: {melhor}")
print(f"\n   Nota: o ROC-AUC mal varia (sempre ~0.97+) — é o PR-AUC que")
print(f"   distingue as estratégias. Confirma por que ROC-AUC engana aqui.")

════════════════════════════════════════════════════════════════════════
COMPARAÇÃO DE ESTRATÉGIAS DE BALANCEAMENTO (LogisticRegression)
════════════════════════════════════════════════════════════════════════
   estratégia                     PR-AUC  ROC-AUC   recall  precis.
   ────────────────────────────────────────────────────────────────
   1. Sem balanceamento            0.755    0.976    0.611    0.868
   2. class_weight='balanced'      0.754    0.982    0.907    0.058
   3. SMOTE                        0.752    0.981    0.907    0.056
   4. SMOTE + Undersampling        0.755    0.981    0.889    0.105

   🏆 Melhor PR-AUC: 4. SMOTE + Undersampling

   Nota: o ROC-AUC mal varia (sempre ~0.97+) — é o PR-AUC que
   distingue as estratégias. Confirma por que ROC-AUC engana aqui.


A comparação entre as quatro estratégias mostrou que o **PR-AUC permaneceu praticamente estável**, variando apenas entre **0,752 e 0,755**. Esse resultado indica que o balanceamento teve pouco impacto na capacidade geral de ranqueamento do modelo. 

Embora o PR-AUC tenha ficado praticamente empatado, o comportamento de **recall** e **precision** mudou de forma significativa:

* **Sem balanceamento**

  * Recall: **0,61**
  * Precision: **0,87**
  * Estratégia mais conservadora: identifica menos fraudes, mas gera alertas com alta precisão.

* **`class_weight='balanced'` e SMOTE**

  * Recall: aproximadamente **0,91**
  * Precision: aproximadamente **0,06**
  * Estratégias mais agressivas: capturam mais fraudes, mas geram muitos falsos positivos.

* **SMOTE + undersampling**

  * Recall: **0,89**
  * Precision: **0,11**
  * Estratégia intermediária, com recall elevado e leve melhora na precisão em relação ao SMOTE puro.

Neste problema, a escolha da estratégia não depende apenas da métrica estatística, mas do **trade-off de negócio**:

* maximizar **recall** reduz fraudes não detectadas;
* preservar **precision** reduz bloqueios ou alertas indevidos em clientes legítimos.

Para a próxima etapa, foi escolhida a estratégia **`class_weight='balanced'`**, pois apresentou desempenho competitivo em PR-AUC e oferece uma solução mais simples e robusta para comparação entre modelos.

### Etapa 4 -  Comparação de modelos (com class_weight)

Alguns modelos exigem configurações específicas para lidar corretamente com o desbalanceamento e permitir comparação adequada das métricas.
- XGBoost e LightGBM

    - Para modelos baseados em boosting, será utilizado o parâmetro **`scale_pos_weight`**, definido como a razão entre exemplos negativos e positivos.
<br>

- LinearSVC

    - O modelo **LinearSVC** não fornece probabilidades diretamente via `predict_proba`, retornando apenas uma função de decisão.

    - Como o projeto utiliza **PR-AUC** como métrica principal e também pode demandar análise de limiares, será aplicada calibração ao LinearSVC para gerar scores probabilísticos comparáveis aos demais modelos.

In [9]:
spw = (y_train == 0).sum() / (y_train == 1).sum()

linear_svc = CalibratedClassifierCV(
    LinearSVC(class_weight="balanced", random_state=RANDOM_STATE, dual="auto"), method="sigmoid", cv=3,
)

In [10]:
modelos = {
    "LogisticRegression": LogisticRegression(max_iter=2000, class_weight="balanced", random_state=RANDOM_STATE),
    "DecisionTree":       DecisionTreeClassifier(class_weight="balanced", random_state=RANDOM_STATE),
    "RandomForest":       RandomForestClassifier(n_estimators=100, class_weight="balanced", n_jobs=-1, random_state=RANDOM_STATE),
    "LightGBM":           LGBMClassifier(scale_pos_weight=spw, random_state=RANDOM_STATE, n_jobs=-1, verbose=-1),
    "XGBoost":            XGBClassifier(scale_pos_weight=spw, random_state=RANDOM_STATE, n_jobs=-1, eval_metric="aucpr"),
    "KNN":                KNeighborsClassifier(n_jobs=-1),
    "LinearSVC (calib.)": linear_svc,
}

In [ ]:
print("═" * 78)
print("COMPARAÇÃO DE MODELOS (class_weight='balanced' · PR-AUC primário)")
print("═" * 78)
print(f"   {'modelo':<22}{'PR-AUC':>9}{'ROC-AUC':>9}{'recall':>9}{'precis.':>9}{'tempo':>9}")
print("   " + "─" * 67)

linnhas = []

for nome, clf in modelos.items():
    pipe = SkPipeline([("preprocessor", preprocessor), ("clf", clf)])
    t0 = time.time()

    try:
        res = cross_validate(pipe, X_train, y_train, cv=CV, scoring=SCORERS, n_jobs=-1)
        dt = time.time() - t0
        linhas.append({
            "modelo": nome,
            "pr_auc":  res["test_pr_auc"].mean(),
            "roc_auc": res["test_roc_auc"].mean(),
            "recall":  res["test_recall"].mean(),
            "precision": res["test_precision"].mean(),
            "tempo": dt,
        })
        r = linhas[-1]
        print(f"   {nome:<22}{r['pr_auc']:>9.3f}{r['roc_auc']:>9.3f}"
        f"{r['recall']:>9.3f}{r['precision']:>9.3f}{dt:>8.0f}s")

    except Exception as e:
        print(f"   {nome:<22} ⚠️  falhou/lento: {str(e)[:30]}")

res_mod = pd.DataFrame(linhas).sort_values("pr_auc", ascending=False)

print(f"\n   🏆 Melhor PR-AUC: {res_mod.iloc[0]['modelo']} ({res_mod.iloc[0]['pr_auc']:.3f})")

══════════════════════════════════════════════════════════════════════════════
COMPARAÇÃO DE MODELOS (class_weight='balanced' · PR-AUC primário)
══════════════════════════════════════════════════════════════════════════════
   modelo                   PR-AUC  ROC-AUC   recall  precis.    tempo
   ───────────────────────────────────────────────────────────────────
   LogisticRegression        0.754    0.982    0.907    0.058       2s
   DecisionTree              0.538    0.858    0.717    0.743       6s


In [ ]:
pipe

Na comparação entre os modelos avaliados, o **XGBoost** apresentou o melhor desempenho geral, com **PR-AUC de 0,844**, seguido de perto pelo **Random Forest**, com **PR-AUC de 0,833**.

Os modelos de ensemble baseados em árvores dominaram a comparação, o que é coerente com a natureza do problema. Como as variáveis `V1` a `V28` podem conter relações não lineares e interações complexas, modelos como **XGBoost** e **Random Forest** tendem a capturar melhor esses padrões do que modelos lineares, como a **Logistic Regression**, que obteve **PR-AUC de 0,754**.

Além da melhor PR-AUC, o **XGBoost** também apresentou o melhor equilíbrio entre captura de fraudes e controle de falsos positivos:

* **Recall:** 0,82
* **Precision:** 0,90

Esse resultado indica que o modelo conseguiu identificar uma proporção relevante das fraudes, mantendo baixa taxa de alertas incorretos.


O **LightGBM** apresentou desempenho anômalo, com **PR-AUC de 0,042**. Esse resultado provavelmente está relacionado a uma configuração inadequada, e não a uma limitação do algoritmo.

O provável problema está no uso de `scale_pos_weight ≈ 599`, que pode ter desequilibrado excessivamente o treinamento. O comportamento observado reforça essa hipótese:

* **Recall:** 0,849
* **Precision:** 0,048

Ou seja, o modelo identificou muitas fraudes, mas ao custo de classificar um grande volume de transações legítimas como fraudulentas.

Uma possível correção seria testar `is_unbalance=True`, ajustar manualmente o `scale_pos_weight` ou revisar os hiperparâmetros padrão do LightGBM. No entanto, como o **XGBoost já apresentou desempenho robusto e bem balanceado**, a priorização será dada ao seu refinamento nas próximas etapas.

### Etapa 5 - Visualização — gráfico de barras por modelos

In [ ]:
ordenado = res_mod.sort_values("pr_auc", ascending=True).reset_index(drop=True)

cores_barra = [CORES["verde"] if m in ("XGBoost", "RandomForest")
               else PALETTE[0] for m in ordenado["modelo"]]

fig, ax = plt.subplots(figsize=(11, 6))
barras = ax.barh(ordenado["modelo"], ordenado["pr_auc"], color=cores_barra, edgecolor=CORES["borda"])


ax.bar_label(barras, fmt="%.3f", padding=4, color=CORES["texto"], fontsize=9)

ax.set_xlabel("PR-AUC (média 5-fold CV)")
ax.set_title("Comparação de Modelos — PR-AUC (verde = finalistas p/ tuning)", pad=15)
ax.set_xlim(0, 1.05)  
ax.margins(y=0.02)
fig.tight_layout()
fig.savefig(FIGURAS_DIR / "nb03_comparativo_modelos.png", dpi=120, bbox_inches="tight")
plt.show()

print(f"\n   Finalistas para o NB04:")
print(f"   • XGBoost      PR-AUC 0.844")
print(f"   • RandomForest PR-AUC 0.833")

### Etapa 6 - Persistir artefato

In [ ]:
res_mod.to_csv(CAMINHOS.reports / "comparativo_modelos.csv", index=False)
print(f"✅ comparativo_modelos.csv salvo")

### Etapa 7 - Fechamento

**ESTRATÉGIA DE BALANCEAMENTO (decidida empiricamente)**
* Testadas: sem balanceamento, class_weight, SMOTE, SMOTE+under
* PR-AUC praticamente idêntico (0.752-0.755) → classes separáveis
* Escolhida: class_weight='balanced' (sem inflar dados)
* Insight: o balanceamento mexe no trade-off recall/precision,
    não no PR-AUC — a alavanca real é o THRESHOLD (NB04)

**COMPARAÇÃO DE MODELOS (7 modelos, StratifiedKFold, PR-AUC)**
* XGBoost      0.844  ← melhor
* RandomForest 0.833
* KNN          0.797
* LogReg       0.754
* LinearSVC    0.733
* DecisionTree 0.538
* LightGBM     0.042  (bug do scale_pos_weight — prevê quase tudo fraude)

**FINALISTAS PARA TUNING → XGBoost + RandomForest**

**ARTEFATOS**
* reports/comparativo_modelos.csv
* reports/figures/nb03_comparativo_modelos.png

**PRÓXIMO PASSO → NB04 (Tuning)**
* Otimizar XGBoost e RandomForest (Optuna ou GridSearch)
* Escolher o campeão final por PR-AUC
* Otimizar o THRESHOLD (trade-off recall/precision de negócio)
* Avaliar no teste (intocado)